# 🛰 Optical Imagery Downloader
Universal downloader for **Sentinel-2** and **Landsat 8/9**.

**Data source priority:** Planetary Computer (free, no auth) → AWS sentinel-cogs / usgs-landsat

---
### Requirements
```bash
pip install pystac-client planetary-computer geopandas shapely
pip install requests contextily matplotlib tqdm
# For efficient clipping: GDAL ≥ 3.2 with /vsis3/ + AWS CLI
```

In [6]:
# !conda install geopandas -c conda-forge -y

# conda  environment : conda activate autoRIFT
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '.')   # adjust if optical_downloader.py is elsewhere

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import MaxNLocator
import warnings
warnings.filterwarnings('ignore')

from optical_downloader import ImageryDownloader, BAND_CONFIG

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

---
## ① Configuration
Set your **AOI**, **sensors**, **date range**, **bands**, and **cloud cover** threshold here.
Everything below runs automatically.

In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║               USER CONFIGURATION                        ║
# ╚══════════════════════════════════════════════════════════╝

# ── AOI: use bbox OR shapefile (set the other to None) ───────────────────────
#  bbox format: [min_lon, min_lat, max_lon, max_lat]  (WGS84)
BBOX      = [151.604, -68.989, 154.178, -68.534]   # Cook Ice Shelf area
SHAPEFILE = None                                    # e.g. 'cook_aoi.shp'

# ── Sensors (list one or both) ────────────────────────────────────────────────
# SENSORS = ['sentinel-2-l2a']  # 'landsat-c2-l2'
# SENSORS = ['sentinel-2-l2a']
SENSORS = ['landsat-c2-l2']

# ── Date range ────────────────────────────────────────────────────────────────
DATE_RANGE = '2022-01-01/2026-03-01'

# ── Cloud cover limit (%) ─────────────────────────────────────────────────────
MAX_CLOUD = 20.0

# ── Bands to download ─────────────────────────────────────────────────────────
# Common names (work for both S2 and Landsat):
#   'blue', 'green', 'red', 'nir', 'nir08', 'swir16', 'swir22',
#   'coastal', 'scl' (S2 cloud mask), 'qa_pixel' (Landsat cloud mask)
DOWNLOAD_BANDS = ['red'] #['red', 'green', 'blue', 'nir']

# ── Clip output to AOI? (recommended — saves disk space) ─────────────────────
CLIP = False

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = './imagery_output'

# ── AWS profile for Landsat (Requester Pays). Set None to skip Landsat AWS. ──
AWS_PROFILE = None    # e.g. 'my_aws_profile'

# ── Max scenes to download (set None for all) ────────────────────────────────
MAX_DOWNLOAD = 5000

# data source preference (for Landsat): "pc" for Planetary Computer, "aws" for AWS (Landsat only)
PREFERRED_SOURCE= "aws"  # "pc" for Planetary Computer, "aws" for AWS (Landsat only), 'auto' to try PC first then AWS

---
## ② Initialise downloader

In [8]:
dl = ImageryDownloader(
    output_dir      = OUTPUT_DIR,
    max_cloud_cover = MAX_CLOUD,
    aws_profile     = AWS_PROFILE,
    use_pc          = True,
    use_aws         = True,
)

---
## ③ Search scenes

In [9]:
results = dl.search(
    sensors    = SENSORS,
    bbox       = BBOX,
    shapefile  = SHAPEFILE,
    date_range = DATE_RANGE,
    max_items  = 5000,
)

# Summary table
dl.summary_table(results)

🔍 Searching ['landsat-c2-l2']  |  2022-01-01/2026-03-01  |  cloud ≤ 10.0%
   landsat-c2-l2: 0 scenes
⚠  No scenes found. Try expanding date range, bbox, or cloud threshold.


""


In [10]:
# dl.download(results.head(4), bands=['red'], clip=False, preferred_source="pc")

---
## ④ Visualise scene footprints

In [11]:
# ── Main footprint map ────────────────────────────────────────────────────────
fig, ax = dl.plot_footprints(
    results,
    color_by    = 'cloud_cover',
    figsize     = (14, 8),
    add_basemap = True,
)
plt.show()

No search results to plot.


In [12]:
# ── Extended analysis dashboard ───────────────────────────────────────────────
if not results.empty:
    fig = plt.figure(figsize=(16, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

    # (A) Footprint map coloured by collection
    ax_map = fig.add_subplot(gs[:, :2])
    colors  = {'sentinel-2-l2a': '#2196F3', 'landsat-c2-l2': '#FF9800'}
    for coll, grp in results.groupby('collection'):
        valid = grp[grp.geometry.notna()]
        if not valid.empty:
            valid.plot(ax=ax_map, alpha=0.25, edgecolor=colors.get(coll, 'gray'),
                       facecolor=colors.get(coll, 'gray'), linewidth=0.7,
                       label=coll)
    if dl._aoi_gdf is not None:
        dl._aoi_gdf.plot(ax=ax_map, facecolor='none', edgecolor='red',
                         linewidth=2.5, linestyle='--', label='AOI')
    try:
        import contextily as ctx
        ctx.add_basemap(ax_map, crs=results.crs.to_string(),
                        source=ctx.providers.Esri.NatGeoWorldMap,
                        attribution_size=5)
    except Exception:
        pass
    patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()
               if k in results['collection'].values]
    patches.append(mpatches.Patch(edgecolor='red', facecolor='none', label='AOI',
                                  linestyle='--'))
    ax_map.legend(handles=patches, loc='upper right', fontsize=9)
    ax_map.set_title('Scene footprints by sensor', fontsize=11)
    ax_map.set_xlabel('Longitude'); ax_map.set_ylabel('Latitude')

    # (B) Cloud cover histogram
    ax_hist = fig.add_subplot(gs[0, 2])
    for coll, grp in results.groupby('collection'):
        cc = grp['cloud_cover'].dropna()
        if not cc.empty:
            ax_hist.hist(cc, bins=15, alpha=0.65, label=coll,
                         color=colors.get(coll, 'gray'), edgecolor='white')
    ax_hist.axvline(MAX_CLOUD, color='red', linewidth=1.5,
                    linestyle='--', label=f'threshold={MAX_CLOUD}%')
    ax_hist.set_xlabel('Cloud cover (%)', fontsize=9)
    ax_hist.set_ylabel('Count', fontsize=9)
    ax_hist.set_title('Cloud cover distribution', fontsize=10)
    ax_hist.legend(fontsize=8)

    # (C) Scene count by month
    ax_ts = fig.add_subplot(gs[1, 2])
    r2 = results.copy()
    r2['month'] = r2['datetime'].dt.to_period('M')
    monthly = r2.groupby(['month', 'collection']).size().unstack(fill_value=0)
    monthly.index = monthly.index.astype(str)
    bottom = np.zeros(len(monthly))
    for coll in monthly.columns:
        ax_ts.bar(monthly.index, monthly[coll], bottom=bottom,
                  label=coll, color=colors.get(coll, 'gray'), alpha=0.85)
        bottom += monthly[coll].values
    ax_ts.set_xticklabels(monthly.index, rotation=45, ha='right', fontsize=7)
    ax_ts.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax_ts.set_ylabel('Scenes', fontsize=9)
    ax_ts.set_title('Scenes per month', fontsize=10)
    ax_ts.legend(fontsize=8)

    plt.suptitle(
        f'Search results: {DATE_RANGE}  |  cloud ≤ {MAX_CLOUD}%  |  n={len(results)}',
        fontsize=12, y=1.01
    )
    plt.show()

---
## ⑤ Filter & inspect before downloading

In [13]:
# ── Filter results (examples — adjust as needed) ──────────────────────────────

# Option A: by sensor
# filtered = results[results['collection'] == 'sentinel-2-l2a'].copy()

# Option B: by date
# filtered = results[results['datetime'] >= '2025-01-01'].copy()

# Option C: lowest cloud cover first
filtered = results.sort_values('cloud_cover').copy()

# Limit to N scenes for download
if MAX_DOWNLOAD:
    to_download = filtered.head(MAX_DOWNLOAD)
else:
    to_download = filtered

print(f"Selected {len(to_download)} scenes for download:")
dl.summary_table(to_download)

KeyError: 'cloud_cover'

In [ ]:
for scene_id in to_download['id']:
    print(scene_id)

LC09_L2SR_076109_20241015_02_T2
LC08_L2SR_074109_20241025_02_T2
LC09_L2SR_074109_20241102_02_T2
LC08_L2SR_076109_20241124_02_T2
LC08_L2SR_078108_20241122_02_T2
LC09_L2SR_077108_20241123_02_T2
LC09_L2SR_076109_20241116_02_T2
LC08_L2SR_075109_20241117_02_T2
LC08_L2SR_077109_20241201_02_T2
LC09_L2SR_076109_20241202_02_T2
LC08_L2SR_077108_20241201_02_T2
LC08_L2SR_078108_20241224_02_T2
LC08_L2SR_076109_20241210_02_T2
LC08_L2SR_075109_20241219_02_T2
LC08_L2SR_075109_20241203_02_T2
LC08_L2SR_075108_20241203_02_T2
LC09_L2SR_075109_20250213_02_T2
LC09_L2SR_077109_20250110_02_T2
LC09_L2SR_077109_20241225_02_T2
LC08_L2SR_074109_20241126_02_T2
LC09_L2SR_076108_20241202_02_T2
LC08_L2SR_076109_20241108_02_T2
LC08_L2SR_074108_20241025_02_T2
LC09_L2SR_075109_20241024_02_T2
LC08_L2SR_074109_20241009_02_T2
LC08_L2SR_074109_20250113_02_T2
LC09_L2SR_074109_20241204_02_T2
LC09_L2SR_077108_20250211_02_T2
LC09_L2SR_077108_20241107_02_T2
LC09_L2SR_074108_20241017_02_T2
LC09_L2SR_074108_20241102_02_T2
LC08_L2S

In [ ]:
# ── Available bands per collection ───────────────────────────────────────────
print("Available bands:")
for coll, bands in BAND_CONFIG.items():
    print(f"\n  {coll}:")
    for name, (pc_key, aws_file) in bands.items():
        print(f"    {name:15s}  (PC: {pc_key:15s}  AWS: {aws_file})")

Available bands:

  sentinel-2-l2a:
    coastal          (PC: B01              AWS: B01.tif)
    blue             (PC: B02              AWS: B02.tif)
    green            (PC: B03              AWS: B03.tif)
    red              (PC: B04              AWS: B04.tif)
    rededge1         (PC: B05              AWS: B05.tif)
    rededge2         (PC: B06              AWS: B06.tif)
    rededge3         (PC: B07              AWS: B07.tif)
    nir              (PC: B08              AWS: B08.tif)
    nir08            (PC: B8A              AWS: B8A.tif)
    nir09            (PC: B09              AWS: B09.tif)
    swir16           (PC: B11              AWS: B11.tif)
    swir22           (PC: B12              AWS: B12.tif)
    scl              (PC: SCL              AWS: SCL.tif)
    visual           (PC: visual           AWS: TCI.tif)
    aot              (PC: AOT              AWS: AOT.tif)
    wvp              (PC: WVP              AWS: WVP.tif)

  landsat-c2-l2:
    coastal          (PC: coastal 

---
## ⑥ Download

In [ ]:
download_results = dl.download(
    items          = to_download,
    bands          = DOWNLOAD_BANDS,
    clip           = CLIP,
    bbox           = BBOX,
    shapefile      = SHAPEFILE,
    skip_existing  = True,
    preferred_source="pc"
)


[1/50] LC09_L2SR_076109_20241015_02_T2  [src=PC → backend=PC]
  ✓  red           (exists)

[2/50] LC08_L2SR_074109_20241025_02_T2  [src=PC → backend=PC]
  ✓  red           (exists)

[3/50] LC09_L2SR_074109_20241102_02_T2  [src=PC → backend=PC]
  ✓  red           (exists)

[4/50] LC08_L2SR_076109_20241124_02_T2  [src=PC → backend=PC]
  ⬇  red           →  LC08_L2SR_076109_20241124_02_T2_red.tif  (81.3 MB)

[5/50] LC08_L2SR_078108_20241122_02_T2  [src=PC → backend=PC]
  ⬇  red           →  LC08_L2SR_078108_20241122_02_T2_red.tif  (92.9 MB)

[6/50] LC09_L2SR_077108_20241123_02_T2  [src=PC → backend=PC]
  ⬇  red           →  LC09_L2SR_077108_20241123_02_T2_red.tif  (93.0 MB)

[7/50] LC09_L2SR_076109_20241116_02_T2  [src=PC → backend=PC]
  ⬇  red           →  LC09_L2SR_076109_20241116_02_T2_red.tif  (83.4 MB)

[8/50] LC08_L2SR_075109_20241117_02_T2  [src=PC → backend=PC]
  ⬇  red           →  LC08_L2SR_075109_20241117_02_T2_red.tif  (82.8 MB)

[9/50] LC08_L2SR_077109_20241201_02_T2  [src=P

---
## ⑦ Quick preview of downloaded images

In [ ]:
import rasterio
from rasterio.plot import show as rshow
import matplotlib.pyplot as plt

# Collect all downloaded tifs
all_files = []
for item_id, bands_dict in download_results.items():
    for band, path in bands_dict.items():
        all_files.append((item_id, band, path))

if not all_files:
    print("No files downloaded.")
else:
    n = min(len(all_files), 9)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten()

    for idx, (item_id, band, path) in enumerate(all_files[:n]):
        ax = axes[idx]
        try:
            with rasterio.open(path) as src:
                data = src.read(1).astype(float)
                data[data == src.nodata] = np.nan if src.nodata is not None else data
            p2, p98 = np.nanpercentile(data, [2, 98])
            ax.imshow(data, vmin=p2, vmax=p98, cmap='gray', interpolation='nearest')
            ax.set_title(f"{item_id[:20]}\n{band}", fontsize=7)
        except Exception as e:
            ax.text(0.5, 0.5, str(e), transform=ax.transAxes,
                    ha='center', va='center', fontsize=8)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    plt.suptitle('Downloaded bands preview', fontsize=12)
    plt.tight_layout()
    plt.show()

# Summary
print("\nDownload summary:")
for item_id, bands_dict in download_results.items():
    print(f"  {item_id}")
    for band, path in bands_dict.items():
        sz = path.stat().st_size / 1e6 if path.exists() else 0
        print(f"    {band:15s} → {path.name}  ({sz:.1f} MB)")

No files downloaded.

Download summary:
  S2B_MSIL2A_20250130T224339_R072_T56DNJ_20250131T013933
  S2B_MSIL2A_20250110T224339_R072_T55DFD_20250111T010847
  S2B_MSIL2A_20250130T224339_R072_T56DMJ_20250131T013933
  S2B_MSIL2A_20250130T224339_R072_T55DFD_20250131T013933
  S2B_MSIL2A_20241215T222409_R129_T56DNJ_20241216T005638
  S2B_MSIL2A_20250124T222419_R129_T56DNJ_20250125T005342
  S2B_MSIL2A_20250110T224339_R072_T56DMJ_20250111T010847
  S2B_MSIL2A_20250106T230339_R015_T55DFD_20250107T002818
  S2B_MSIL2A_20250130T224339_R072_T56DMK_20250131T013933
  S2B_MSIL2A_20250106T230339_R015_T56DMJ_20250107T002818
  S2B_MSIL2A_20241101T224339_R072_T55DFD_20241101T235847


---
## ⑧ Reproject to EPSG:3031 (Antarctic Polar Stereographic) — optional

In [ ]:
import subprocess
from pathlib import Path

def reproject_3031(src_path: Path, overwrite: bool = False) -> Path:
    dst_path = src_path.with_stem(src_path.stem + '_3031')
    if dst_path.exists() and not overwrite:
        print(f'  exists: {dst_path.name}')
        return dst_path
    cmd = (
        f'gdalwarp -overwrite -t_srs EPSG:3031 '
        f'-r bilinear -co COMPRESS=LZW '
        f'"{src_path}" "{dst_path}"'
    )
    r = subprocess.run(cmd, shell=True, capture_output=True)
    if r.returncode == 0:
        print(f'  reprojected: {dst_path.name}')
        return dst_path
    else:
        print(f'  FAILED: {r.stderr.decode()[-200:]}')
        return None

# Apply to all downloaded files
REPROJECT = False   # set True to enable
if REPROJECT:
    for item_id, bands_dict in download_results.items():
        for band, path in bands_dict.items():
            reproject_3031(path)